In [1]:
import pandas as pd 



ruta = "../../data/fuentes/climaticos/Base_desastres_con_costoyPIB.xlsx"
df = pd.read_excel(ruta)


In [2]:
import os
import pymysql
from pymysql.constants import CLIENT
from dotenv import load_dotenv

load_dotenv()

# Config DB
DB_HOST     = os.getenv('DB_HOST')
DB_USER     = os.getenv('DB_USER', 'root')
DB_PASSWORD = os.getenv('DB_PASSWORD', 'root')
DB_NAME     = os.getenv('DB_NAME', 'tfm_cambio_climatico')

# Conexión
conexion = pymysql.connect(
    host=DB_HOST,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME,
    client_flag=CLIENT.MULTI_STATEMENTS
)
cursor = conexion.cursor()

# 🔹 Filtrar solo los indicadores que empiezan con "costos"
df_sql = df[df["indicador_code"].str.lower().str.startswith("costos")].rename(
    columns={
        "Year": "anio",
        "Value": "valor"
    }
)[["pais_id", "anio", "indicador_id", "valor"]]

total = len(df_sql)
batch_size = 1000
print(f"Total registros a insertar: {total}")

# Insertar en lotes
sql = """
INSERT INTO hechos (pais_id, anio, indicador_id, valor)
VALUES (%s, %s, %s, %s)
"""

for i in range(0, total, batch_size):
    chunk = df_sql.iloc[i:i+batch_size]
    data = list(chunk.itertuples(index=False, name=None))
    cursor.executemany(sql, data)
    conexion.commit()
    print(f"  ✔ Filas insertadas {i+1}–{min(i+batch_size, total)}")

print("✅ Inserción de indicadores 'costos' completada.")


Total registros a insertar: 2107
  ✔ Filas insertadas 1–1000
  ✔ Filas insertadas 1001–2000
  ✔ Filas insertadas 2001–2107
✅ Inserción de indicadores 'costos' completada.
